# 🧭 Chapter 05-03a: Production RAG — Evaluation & Debugging

---

## 📖 The Story: Ship It... But Does It Actually Work?

Your RAG pipeline answers questions. But HOW do you know the answers are good? A wrong answer about medication dosage, financial data, or legal terms could be catastrophic.

Enter RAG evaluation: systematic measurement of quality.

---

## 🎯 RAG Evaluation Metrics

```
╔══════════════════════════════════════════════════════════════════════╗
║  THE RAG EVALUATION TRIANGLE                                        ║
║                                                                      ║
║          Faithfulness                                                ║
║           (Is the answer                                             ║
║            grounded in context?)                                     ║
║               / \                                                    ║
║              /   \                                                   ║
║    Relevance      Context Precision                                  ║
║  (Does it answer   (Are retrieved                                    ║
║   the question?)    chunks relevant?)                                ║
╚══════════════════════════════════════════════════════════════════════╝
```

| Metric | What It Measures | Score Range |
|--------|-----------------|-------------|
| **Faithfulness** | Answer is supported by context (no hallucination) | 0-1 |
| **Answer Relevance** | Answer addresses the question | 0-1 |
| **Context Precision** | Retrieved chunks are relevant | 0-1 |
| **Context Recall** | All needed info was retrieved | 0-1 |

---

## 📊 RAGAS Framework

```
╔══════════════════════════════════════════════════════════╗
║  RAGAS (Retrieval Augmented Generation Assessment)      ║
║                                                          ║
║  Open-source framework for evaluating RAG pipelines.    ║
║  Uses LLMs to judge quality (LLM-as-a-judge).          ║
║                                                          ║
║  Input per example:                                      ║
║  • question: What the user asked                        ║
║  • answer: What your RAG produced                       ║
║  • contexts: What chunks were retrieved                 ║
║  • ground_truth: The correct answer (optional)          ║
╚══════════════════════════════════════════════════════════╝
```

```python
# RAGAS usage pattern
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

result = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision]
)
```

---

## 🐛 Common RAG Failure Modes

### 1. Wrong Retrieval
```
Q: "What's the refund policy?" → Retrieved: Pricing docs instead of refund policy
Fix: Better embeddings, hybrid search, check chunking boundaries
```

### 2. Missing Context (Low Recall)
```
Q: "Full refund policy" → Retrieved only 1 of 3 relevant chunks
Fix: Increase k, multi-query retrieval, parent-child chunking
```

### 3. Hallucination Despite Context
```
Context says $99/mo → Answer says $79/mo
Fix: Stronger prompt instructions, lower temperature, larger model
```

### 4. Context Window Overflow
```
Too many chunks → exceeds model's context limit
Fix: Reduce k, summarize chunks, use map-reduce
```

### 5. Stale Data
```
Docs updated but embeddings not refreshed
Fix: Incremental indexing pipeline, versioned embeddings
```

---

## 🔒 Production Concerns

### Metadata Filtering & Access Control
```python
# Multi-tenant RAG: each user only sees their data
results = vectorstore.similarity_search(
    query,
    filter={"user_id": current_user.id}
)
```

### Monitoring with LangSmith
```
Every RAG call is traced:
├── Input question
├── Retrieved chunks (with scores)
├── Prompt sent to LLM
├── LLM response
├── Latency per step
└── Token usage & cost
```

### Caching
```
Exact cache: Same question → same answer (fast)
Semantic cache: Similar question → cached answer (smart)
```

---

## 🎓 Interview Corner

**Q: Design an enterprise RAG system for a 1000-employee company with 50,000 documents.**

<details>
<summary>💡 Click for Expert Answer</summary>

**Architecture:**
1. **Ingestion pipeline**: Scheduled ETL that crawls Confluence/SharePoint → chunks → embeds → stores
2. **Vector store**: Pinecone with namespaces per department (access control)
3. **Retrieval**: Hybrid search (vector + BM25) → Cross-encoder re-ranking
4. **Generation**: GPT-4o with structured system prompt, citations required
5. **Evaluation**: RAGAS metrics on golden dataset, weekly quality audits
6. **Monitoring**: LangSmith traces, latency alerts, hallucination detection
7. **Security**: Row-level access control via metadata, PII redaction, audit logs
8. **Caching**: Redis semantic cache for frequent queries (80% hit rate target)

**Scale:**
- 50K docs × ~10 chunks each = 500K vectors
- Pinecone handles this easily with <100ms p99 latency
- Cost: ~$70/mo for Pinecone + LLM costs

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Evaluate with: Faithfulness, Relevance, Context Precision.     │
├─────────────────────────────────────────────────────────────────────┤
│  📌 RAGAS framework automates RAG evaluation with LLM-as-judge.   │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Debug retrieval FIRST — most RAG failures are retrieval gaps.  │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Metadata filtering = multi-tenant access control.              │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Monitor everything: LangSmith traces + RAGAS metrics.          │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Enterprise RAG Knowledge Bot Project!

Build a full production RAG system with evaluation.

---

### 🎉 You Can Ship RAG to Production!
Evaluation, debugging, monitoring — the complete production toolkit. 🚀